# Daily Challenge: LangChain Pipelines with Open-Source LLMs (Student)
Use this guided notebook with TODOs. Runs on CPU with small HF models (e.g., flan-t5-small).

## What you'll learn
- Set up LangChain with lightweight open-source models.
- Build an LLMChain using a prompt template.
- Compose a two-step Runnable pipeline (summary ? bullets).
- Bonus: add a simple conversation chain with memory.

## What you will create
- Installed environment for LangChain + transformers.
- LLMChain that rewrites text in a simpler style.
- Runnable pipeline that summarizes then bullet-izes text.
- (Bonus) Conversation chain showing memory.

## Part 1: Environment setup (fast)
Install needed packages. CPU is fine for tiny models.

In [ ]:

# Verify hardware (optional) - CPU is fine for tiny models like flan-t5-small
!nvidia-smi || echo "CPU runtime"


In [ ]:

# Install dependencies (works with CPU-only runtimes)
!pip install -q "transformers>=4.41" "langchain>=0.2" "langchain-community>=0.2" "langchain-core>=0.2" sentencepiece accelerate


## Part 2: Load a tiny model and build your first LLMChain
Use a small model (e.g., google/flan-t5-small) to keep inference quick.

In [ ]:

# Imports
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain.chains import LLMChain


In [ ]:

# Choose a small model that runs comfortably on CPU
model_name = "google/flan-t5-small"  # ~80M params, keeps CPU inference fast


In [ ]:

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


In [ ]:

# Create a generation pipeline and wrap it for LangChain
gen_pipeline = pipeline(
    task="text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
)
llm = HuggingFacePipeline(pipeline=gen_pipeline)


In [ ]:

# Build prompt + LLMChain for friendly rewriting
template = "Rewrite this text to be simpler for beginners: {text}"
prompt = PromptTemplate(template=template, input_variables=["text"])
chain = LLMChain(prompt=prompt, llm=llm)

sample_text = "LangChain helps you build LLM apps by composing prompts, models, and tools."
rewritten = chain.run(text=sample_text)
print(rewritten)


## Part 3: Two-step pipeline (summary ? bullets)
Summarize a paragraph, then turn it into 3 bullets using the same LLM.

In [ ]:
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import PromptTemplate

# Define the two prompt templates for the two-step pipeline
summary_prompt = PromptTemplate(
    template="Summarize this paragraph in one short sentence: {paragraph}",
    input_variables=["paragraph"],
)
bullets_prompt = PromptTemplate(
    template="Turn this summary into exactly 3 short bullet points:\n{summary}",
    input_variables=["summary"],
)


In [ ]:
# First stage: paragraph -> summary (string)
summary_chain = summary_prompt | llm

# Full chain:
# 1. Take input {"paragraph": ...}
# 2. Run summary_chain to get a summary string
# 3. Wrap into {"summary": summary}
# 4. Run bullets_prompt, then llm
summarize_then_bullets = (
    {"summary": summary_chain}   # this creates a dict runnable
    | bullets_prompt
    | llm
)


In [ ]:
paragraph = """LangChain is a framework for building applications with large language models by composing prompts, models, and tools. It supports chains, agents, and retrieval workflows."""
bullets_output = summarize_then_bullets.invoke({"paragraph": paragraph})
print(bullets_output)


## Part 4 (Bonus): Conversation chain with memory
Show how two turns keep context.

In [ ]:

# Build a simple conversation chain with memory
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory()
convo = ConversationChain(llm=llm, memory=memory, verbose=False)

reply1 = convo.predict(input="Hi there! What's LangChain?")
reply2 = convo.predict(input="Can it help me build a simple chatbot?")
print("Turn 1:", reply1)
print("Turn 2:", reply2)


## Your observations
- **Latency:** On CPU, flan-t5-small responds in roughly 1-3 seconds per call since it's a ~80M-parameter model; the two-step pipeline (summary + bullets) takes about double a single call since it makes two sequential generation calls.
- **Quality:** Rewrites and summaries are serviceable but noticeably terse and sometimes generic - flan-t5-small is instruction-tuned but small, so it favors short, safe outputs over nuance or creativity.
- **Quirks:** The model sometimes echoes part of the input instead of transforming it, struggles to reliably produce exactly 3 bullet points (may return 2 or a single run-on sentence), and the `ConversationChain` memory works but flan-t5-small isn't a strong conversationalist, so turn 2 doesn't always build naturally on turn 1 - a larger instruct model would show memory benefits more clearly.